## Generate from trained model

- The model checkpoint used: trained in MacBook Pro (M4 Pro) Chips for approx. 1hr. 
- Batch size = 32, steps = 4200, context length = 256. 
- Therefore the total tokens processed in this training is `34,406,400`
- The loss went down to approx 1.49 at the end of this training (considering the vocab size is only 10k)

## 使用训练好的模型生成文本

### 模型训练信息
- 使用的模型检查点：在MacBookPro(M4Pro)芯片上训练约1小时
- 批次大小 = 32，训练步数 = 4200，上下文长度 = 256
- 因此训练过程中处理的总token数为`34, 406, 400`
- 训练结束时损失降至约1.49（考虑到词表大小只有10k）


### 深度学习小知识
- 模型检查点（Model Checkpoint）：训练过程中保存的模型参数
- 训练步数（Training Steps）：模型更新的次数
- 上下文长度（Context Length）：模型一次能处理的序列长度
- 总token数：训练过程中处理的所有token的总和
- 损失（Loss）：衡量模型预测与真实标签差异的指标

In [ ]:
# 从训练好的模型进行推理
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "main":
    PROJECT_ROOT = PROJECT_ROOT.parent

MAIN_DIR = PROJECT_ROOT / "main"
if str(MAIN_DIR) not in sys.path:
    sys.path.insert(0, str(MAIN_DIR))

from run_train_model import generate
from model import Transformer as Model
from tokenizer_optimized import Tokenizer

# 请替换为你的checkpoint路径
model_ckpt = PROJECT_ROOT / "train_logs" / "run_xxx" / "ckpt_iter_x.pt"
# 词表文件路径
vocab_file = PROJECT_ROOT / "trained_tokenizer" / "vocab_of_your_tokenizer.json"
# 合并规则文件路径
merge_file = PROJECT_ROOT / "trained_tokenizer" / "merges_of_your_tokenizer.json"

# 自动检测可用设备
# 深度学习小知识：
#   - CUDA：NVIDIA GPU计算平台
#   - MPS：Apple Silicon GPU计算平台
#   - CPU：中央处理器（备用）
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"using device: {device}")

# 初始化分词器
# 深度学习小知识：分词器将文本转换为token ID序列
tokenizer = Tokenizer.from_files(str(vocab_file), str(merge_file), special_tokens=["<|endoftext|>"])
# 获取结束符token ID
eos_id = tokenizer.special_token_to_id.get("<|endoftext|>")

# 重建模型
# 深度学习小知识：需要使用与训练时相同的模型参数。
# 如果checkpoint训练时用了 --residual_type attention，这里也必须改成 residual_type="attention"。
model_args = dict(
    vocab_size=10000,  # 词表大小
    context_length=256,  # 上下文长度
    n_head=16,  # 注意力头数
    num_layers=4,  # Transformer层数
    d_model=512,  # 模型维度
    d_ff=1344,  # 前馈网络维度
    theta=10000.0,  # RoPE频率参数
    residual_type="standard",  # 可选："standard" 或 "attention"
    attn_residual_window=0  # 0表示注意力残差可回看全部历史层
)

print(f"Loading model from {model_ckpt}...")
# 创建模型实例并移动到指定设备
model = Model(**model_args).to(device)

# 加载模型权重
# 深度学习小知识：检查点包含模型参数、优化器状态和训练进度
checkpoint = torch.load(model_ckpt, map_location=device)
state_dict = checkpoint['model']  # 检查点包含："model", "optimizer", "iteration"
# 加载模型参数
model.load_state_dict(state_dict)
# 设置为评估模式（影响Dropout、BatchNorm等）
model.eval()

# 测试模型生成能力
# 深度学习小知识：
#   - Prompt（提示）：给模型的输入文本
#   - Temperature（温度）：控制生成的随机性，<1.0更确定，>1.0更随机
#   - Top-p：从累积概率达到p的最小token集合中采样
prompts = [
    "Once upon a time, there was a little boy named Tim.",
    "The cat sat on the mat and",
    "Lily found a magic wand in the garden, and she"
]

print("-" * 30)
# 对每个提示词生成文本
for p in prompts:
    print(f"Prompt: {p}")
    # 生成文本
    # 参数说明：
    #   - context: 输入提示词
    #   - max_new_tokens: 最大生成token数
    #   - temperature: 温度参数（0.8表示适中的随机性）
    #   - top_p: Top-p采样阈值（0.9表示从累积概率达到0.9的token集合中采样）
    #   - eos_id: 结束符token ID
    #   - context_length: 模型支持的最大上下文长度
    #   - device: 计算设备
    full_output, _ = generate(model, tokenizer=tokenizer, context=p, max_new_tokens=256, temperature=0.8, top_p=0.9,
                              eos_id=eos_id, context_length=256, device=device)
    print(f"Generated: {full_output}")
    print("=" * 80)